In [18]:
import torch
from torch import nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Using device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
Memory: 6.44 GB


In [19]:
data_dir = "../data/chest_xray"
train_path = os.path.join(data_dir, "train")
test_path = os.path.join(data_dir, "test")

# Defining the transforms for the datasets
train_transforms = transforms.Compose([
    transforms.Resize((224, 244)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 244)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Build two views of the same training folder so train and val can use different transforms.
train_full_dataset = datasets.ImageFolder(train_path, transform=train_transforms)
val_full_dataset = datasets.ImageFolder(train_path, transform=val_transforms)

# Reproducible 80-20 split by indices.
generator = torch.Generator().manual_seed(42)
indices = torch.randperm(len(train_full_dataset), generator=generator).tolist()
split_idx = int(0.8 * len(indices))
train_indices = indices[:split_idx]
val_indices = indices[split_idx:]

train_dataset = Subset(train_full_dataset, train_indices)
val_dataset = Subset(val_full_dataset, val_indices)

# Test dataset remains original (624 images)
test_dataset = datasets.ImageFolder(test_path, transform=val_transforms)

# Preparing the dataloaders for our datasets
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=16, num_workers=2)
val_dataloader = DataLoader(val_dataset, shuffle=False, batch_size=16, num_workers=2)
test_dataloader = DataLoader(test_dataset, shuffle=False, batch_size=16, num_workers=2)

print(f"Train size: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Train size: 4172 | Val: 1044 | Test: 624


In [20]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Step 1: Freeze everything
for param in model.parameters():
    param.requires_grad = False

# Step 2: Unfreeze BatchNorm layers everywhere
for module in model.modules():
    if isinstance(module, nn.BatchNorm2d):
        for param in module.parameters():
            param.requires_grad = True

# Step 3: Unfreeze the LAST ResNet block (layer4)
# ResNet18 structure: conv1, bn1, relu, maxpool, layer1, layer2, layer3, layer4, avgpool, fc
for param in model.layer4.parameters():
    param.requires_grad = True

# Step 4: Replace FC head
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 2)
)

In [21]:
model = model.to(device)

# Simple approach: collect all trainable params
trainable_params = filter(lambda p: p.requires_grad, model.parameters())
optimizer = optim.Adam(trainable_params, lr=0.001, weight_decay=1e-4)

criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

In [22]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)
        _, predicted = torch.max(pred, 1)
        total += y.size(0)
        correct += (predicted == y).sum().item()

        epoch_loss = running_loss / total
        epoch_acc = correct / total
        return epoch_loss, epoch_acc
    
def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

num_epochs = 25
best_val_acc = 0.0

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_dataloader, criterion, optimizer, device)
    val_loss, val_acc = validate_epoch(model, val_dataloader, criterion, device)
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '../backend/models/resnet18_pneumonia_best.pth')
        print(f"** Saved best model with val acc: {val_acc:.4f} **")
    
    print("-" * 40)

Epoch 1/25
Train Loss: 0.9918 | Train Acc: 0.1875
Val Loss: 0.5043 | Val Acc: 0.7328
** Saved best model with val acc: 0.7328 **
----------------------------------------
Epoch 2/25
Train Loss: 0.5538 | Train Acc: 0.7500
Val Loss: 0.4495 | Val Acc: 0.7289
----------------------------------------
Epoch 3/25
Train Loss: 0.7355 | Train Acc: 0.6250
Val Loss: 0.2676 | Val Acc: 0.8582
** Saved best model with val acc: 0.8582 **
----------------------------------------
Epoch 4/25
Train Loss: 0.2501 | Train Acc: 0.9375
Val Loss: 0.2181 | Val Acc: 0.9033
** Saved best model with val acc: 0.9033 **
----------------------------------------
Epoch 5/25
Train Loss: 0.2640 | Train Acc: 0.8750
Val Loss: 0.1611 | Val Acc: 0.9406
** Saved best model with val acc: 0.9406 **
----------------------------------------
Epoch 6/25
Train Loss: 0.3128 | Train Acc: 0.7500
Val Loss: 0.1365 | Val Acc: 0.9454
** Saved best model with val acc: 0.9454 **
----------------------------------------
Epoch 7/25
Train Loss: 0

In [23]:
import torch

checkpoint = torch.load('../backend/models/resnet18_pneumonia_best.pth', map_location=device)
model.load_state_dict(checkpoint)
test_loss, test_acc = validate_epoch(model, test_dataloader, criterion, device)
print(f"Final Test Accuracy: {test_acc:.4f}")  # Should be ~85-90%

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_20560\3931284602.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('../backend/models/resnet18_pneumon

Final Test Accuracy: 0.7612


In [24]:
# Save complete package for FastAPI (uses tuned threshold if available).
decision_threshold = float(optimal_threshold) if 'optimal_threshold' in globals() else 0.5

torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': ['NORMAL', 'PNEUMONIA'],
    'input_size': [224, 244],
    'normalization': {
        'mean': [0.485, 0.456, 0.406],
        'std': [0.229, 0.224, 0.225]
    },
    'decision_threshold': decision_threshold,
    'architecture': 'resnet18_custom'
}, '../backend/models/pneumonia_model_full.pt')

print(f"Model saved for FastAPI deployment with threshold: {decision_threshold:.4f}")

Model saved for FastAPI deployment with threshold: 0.6407


In [25]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_dataloader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=['NORMAL', 'PNEUMONIA']))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(cm)

Classification Report:
              precision    recall  f1-score   support

      NORMAL       0.96      0.38      0.54       234
   PNEUMONIA       0.73      0.99      0.84       390

    accuracy                           0.76       624
   macro avg       0.84      0.69      0.69       624
weighted avg       0.81      0.76      0.73       624

Confusion Matrix:
[[ 89 145]
 [  4 386]]


In [26]:
from sklearn.metrics import precision_recall_curve

# Get validation probabilities
model.eval()
val_probs = []
val_labels = []

with torch.no_grad():
    for images, labels in val_dataloader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        val_probs.extend(probs[:, 1].cpu().numpy())  # probability of PNEUMONIA
        val_labels.extend(labels.cpu().numpy())

val_probs = np.array(val_probs)
val_labels = np.array(val_labels)

# Compute precision-recall curve
precisions, recalls, thresholds = precision_recall_curve(val_labels, val_probs)

# Find threshold that gives high precision (fewer false positives) while maintaining decent recall
# Let's target precision >= 0.9
target_precision = 0.9
idx = np.where(precisions[:-1] >= target_precision)[0]
if len(idx) > 0:
    optimal_threshold = thresholds[idx[0]]
    print(f"Optimal threshold for precision >= {target_precision}: {optimal_threshold:.4f}")
else:
    # Fallback: use threshold at max F1
    from sklearn.metrics import f1_score
    f1_scores = [f1_score(val_labels, val_probs >= t) for t in thresholds]
    optimal_threshold = thresholds[np.argmax(f1_scores)]
    print(f"Optimal threshold by F1: {optimal_threshold:.4f}")

Optimal threshold for precision >= 0.9: 0.1551


In [27]:
test_probs = []
test_labels = []
with torch.no_grad():
    for images, labels in test_dataloader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        test_probs.extend(probs[:, 1].cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

new_preds = (np.array(test_probs) >= optimal_threshold).astype(int)
print("With threshold", optimal_threshold)
print(classification_report(test_labels, new_preds, target_names=['NORMAL', 'PNEUMONIA']))

With threshold 0.15509401
              precision    recall  f1-score   support

      NORMAL       0.98      0.21      0.34       234
   PNEUMONIA       0.68      1.00      0.81       390

    accuracy                           0.70       624
   macro avg       0.83      0.60      0.57       624
weighted avg       0.79      0.70      0.63       624



In [31]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

threshold_to_use = float(optimal_threshold) if 'optimal_threshold' in globals() else 0.5
print(f"Using tuned threshold: {threshold_to_use:.4f}")

def collect_probs_and_labels(model, loader, device):
    model.eval()
    probs_pos = []
    labels_all = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.nn.functional.softmax(outputs, dim=1)[:, 1]
            probs_pos.extend(probs.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    return np.array(probs_pos), np.array(labels_all)

def evaluate_at_threshold(probs_pos, labels, threshold, split_name):
    preds = (probs_pos >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    metrics = {
        'split': split_name,
        'threshold': threshold,
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds, zero_division=0),
        'recall': recall_score(labels, preds, zero_division=0),
        'f1': f1_score(labels, preds, zero_division=0),
        'specificity': specificity,
        'roc_auc': roc_auc_score(labels, probs_pos),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }

    print(f"\n[{split_name}] Threshold={threshold:.4f}")
    print(
        f"Acc: {metrics['accuracy']:.4f} | Precision: {metrics['precision']:.4f} | "
        f"Recall: {metrics['recall']:.4f} | F1: {metrics['f1']:.4f} | "
        f"Specificity: {metrics['specificity']:.4f} | ROC-AUC: {metrics['roc_auc']:.4f}"
    )
    print(f"Confusion Matrix -> TN:{tn} FP:{fp} FN:{fn} TP:{tp}")
    print(classification_report(labels, preds, target_names=['NORMAL', 'PNEUMONIA']))

    return metrics

val_probs_full, val_labels_full = collect_probs_and_labels(model, val_dataloader, device)
test_probs_full, test_labels_full = collect_probs_and_labels(model, test_dataloader, device)

val_metrics = evaluate_at_threshold(val_probs_full, val_labels_full, threshold_to_use, 'Validation')
test_metrics = evaluate_at_threshold(test_probs_full, test_labels_full, threshold_to_use, 'Test')

Using tuned threshold: 0.1551

[Validation] Threshold=0.1551
Acc: 0.9828 | Precision: 0.9869 | Recall: 0.9895 | F1: 0.9882 | Specificity: 0.9647 | ROC-AUC: 0.9991
Confusion Matrix -> TN:273 FP:10 FN:8 TP:753
              precision    recall  f1-score   support

      NORMAL       0.97      0.96      0.97       283
   PNEUMONIA       0.99      0.99      0.99       761

    accuracy                           0.98      1044
   macro avg       0.98      0.98      0.98      1044
weighted avg       0.98      0.98      0.98      1044


[Test] Threshold=0.1551
Acc: 0.7676 | Precision: 0.7307 | Recall: 0.9949 | F1: 0.8426 | Specificity: 0.3889 | ROC-AUC: 0.9634
Confusion Matrix -> TN:91 FP:143 FN:2 TP:388
              precision    recall  f1-score   support

      NORMAL       0.98      0.39      0.56       234
   PNEUMONIA       0.73      0.99      0.84       390

    accuracy                           0.77       624
   macro avg       0.85      0.69      0.70       624
weighted avg       0.

In [28]:
# Save deployment package with calibrated decision threshold (if computed above).
decision_threshold = float(optimal_threshold) if 'optimal_threshold' in globals() else 0.5

torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': ['NORMAL', 'PNEUMONIA'],
    'input_size': [224, 244],
    'normalization': {
        'mean': [0.485, 0.456, 0.406],
        'std': [0.229, 0.224, 0.225]
    },
    'decision_threshold': decision_threshold,
    'architecture': 'resnet18_custom'
}, '../backend/models/pneumonia_model_full.pt')

print(f"Saved deployment package with threshold: {decision_threshold:.4f}")

Saved deployment package with threshold: 0.1551


In [29]:
# Only for training - oversample Normal class
from torch.utils.data import WeightedRandomSampler

# Get labels of training dataset
train_labels = [label for _, label in train_dataset]
class_counts = np.bincount(train_labels)
class_weights = 1. / class_counts
sample_weights = class_weights[train_labels]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
train_loader = DataLoader(train_dataset, batch_size=16, sampler=sampler, num_workers=2)

In [30]:
from sklearn.metrics import accuracy_score, recall_score, f1_score


def run_epoch_with_metrics(model, loader, criterion, device, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.set_grad_enabled(is_train):
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            if is_train:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_train:
                loss.backward()
                optimizer.step()

            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_recall = recall_score(all_labels, all_preds, average='binary', pos_label=1)
    epoch_f1 = f1_score(all_labels, all_preds, average='binary', pos_label=1)

    return epoch_loss, epoch_acc, epoch_recall, epoch_f1


# Use weighted sampler loader if it exists, otherwise fall back to the original train dataloader.
active_train_loader = train_loader if 'train_loader' in globals() else train_dataloader

num_epochs = 10
best_val_f1 = 0.0

for epoch in range(num_epochs):
    train_loss, train_acc, train_recall, train_f1 = run_epoch_with_metrics(
        model, active_train_loader, criterion, device, optimizer=optimizer
    )

    val_loss, val_acc, val_recall, val_f1 = run_epoch_with_metrics(
        model, val_dataloader, criterion, device, optimizer=None
    )

    scheduler.step(val_loss)

    print(f"Epoch {epoch + 1}/{num_epochs}")
    print(
        f"Train -> Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | "
        f"Recall: {train_recall:.4f} | F1: {train_f1:.4f}"
    )
    print(
        f"Val   -> Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | "
        f"Recall: {val_recall:.4f} | F1: {val_f1:.4f}"
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), '../backend/models/resnet18_pneumonia_best_finetuned.pth')
        print(f"Saved best fine-tuned model (Val F1: {best_val_f1:.4f})")

    print('-' * 50)

print(f"Best validation F1: {best_val_f1:.4f}")

Epoch 1/10
Train -> Loss: 0.1425 | Acc: 0.9480 | Recall: 0.9553 | F1: 0.9482
Val   -> Loss: 0.0627 | Acc: 0.9722 | Recall: 0.9671 | F1: 0.9807
Saved best fine-tuned model (Val F1: 0.9807)
--------------------------------------------------
Epoch 2/10
Train -> Loss: 0.0780 | Acc: 0.9724 | Recall: 0.9699 | F1: 0.9721
Val   -> Loss: 0.0864 | Acc: 0.9674 | Recall: 0.9580 | F1: 0.9772
--------------------------------------------------
Epoch 3/10
Train -> Loss: 0.0649 | Acc: 0.9789 | Recall: 0.9765 | F1: 0.9792
Val   -> Loss: 0.0702 | Acc: 0.9722 | Recall: 0.9671 | F1: 0.9807
--------------------------------------------------
Epoch 4/10
Train -> Loss: 0.0456 | Acc: 0.9856 | Recall: 0.9841 | F1: 0.9855
Val   -> Loss: 0.0381 | Acc: 0.9837 | Recall: 0.9908 | F1: 0.9889
Saved best fine-tuned model (Val F1: 0.9889)
--------------------------------------------------
Epoch 5/10
Train -> Loss: 0.0335 | Acc: 0.9895 | Recall: 0.9884 | F1: 0.9894
Val   -> Loss: 0.0453 | Acc: 0.9799 | Recall: 0.9790 | F1

In [33]:
summary_keys = ['threshold', 'accuracy', 'precision', 'recall', 'f1', 'specificity', 'roc_auc', 'tn', 'fp', 'fn', 'tp']
print('Validation metrics:')
for k in summary_keys:
    print(f"  {k}: {val_metrics[k]}")

print('\nTest metrics:')
for k in summary_keys:
    print(f"  {k}: {test_metrics[k]}")

Validation metrics:
  threshold: 0.15509401261806488
  accuracy: 0.9827586206896551
  precision: 0.9868938401048493
  recall: 0.9894875164257556
  f1: 0.9881889763779528
  specificity: 0.9646643109540636
  roc_auc: 0.9991316985740354
  tn: 273
  fp: 10
  fn: 8
  tp: 753

Test metrics:
  threshold: 0.15509401261806488
  accuracy: 0.7676282051282052
  precision: 0.7306967984934086
  recall: 0.9948717948717949
  f1: 0.8425624321389794
  specificity: 0.3888888888888889
  roc_auc: 0.963434144203375
  tn: 91
  fp: 143
  fn: 2
  tp: 388
